# Content-Based Filtering Evaluation

## Dataset
`data-contentbased/ecommerce_products_5k.csv` — 5,000 products · 20 sub-categories · 250 products each

## Embed text (mirrors real system)
```
"{name} | {sub_category}"
```
Model: **OpenAI text-embedding-3-small** (1536-dim) — same as production.

## Ground truth
Products that share the same **`sub_category`** are considered relevant (249 per query).

## Metrics
| Metric | Meaning |
|---|---|
| **Precision@K** | Fraction of top-K results in the same sub_category |
| **Recall@K** | Fraction of all 249 relevant items that appear in top-K |
| **HitRate@K** | % of products where ≥1 relevant item appears in top-K |
| **Mean Cosine Similarity** | Average similarity score of top-K results (self-consistency) |

> **Note on Mean Cosine Similarity:** measures how "focused" the recommendations are around the query vector — not whether they are correct. Use alongside Precision/Recall.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset      : {PROJECT_ROOT / 'data-contentbased' / 'ecommerce_products_5k.csv'}")

In [ ]:
import logging
import os
import time
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")

from app.evaluation.cb_evaluator import (
    DEFAULT_CACHE_DIR,
    DEFAULT_K_VALUES,
    EMBED_MODEL,
    RELEVANCE_COL,
    MetricResult,
    build_embed_texts,
    embed_products,
    evaluate,
    load_products,
    print_results,
    run_evaluation,
)

# Load .env for OPENAI_API_KEY
load_dotenv(PROJECT_ROOT / ".env")
api_key = os.getenv("OPENAI_API_KEY")
assert api_key, "OPENAI_API_KEY not found in .env!"
print(f"API key loaded: sk-...{api_key[-6:]}")
print(f"Embed model  : {EMBED_MODEL}")

## 1. Dataset overview

In [ ]:
DATA_DIR = PROJECT_ROOT / "data-contentbased"
df = load_products(DATA_DIR)

print(f"Total products : {len(df):,}")
print(f"Sub-categories : {df[RELEVANCE_COL].nunique()}")
print(f"Products/cat   : {len(df) // df[RELEVANCE_COL].nunique()}")
print()
print("Sample embed texts (first 5):")
for t in build_embed_texts(df)[:5]:
    print(f"  '{t}'")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Products per sub-category
ax = axes[0]
cat_counts = df[RELEVANCE_COL].value_counts()
ax.bar(range(len(cat_counts)), cat_counts.values, color="steelblue", edgecolor="white")
ax.set_xticks(range(len(cat_counts)))
ax.set_xticklabels(cat_counts.index, rotation=45, ha="right", fontsize=8)
ax.set_title("Products per Sub-Category", fontsize=13)
ax.set_ylabel("Count")
ax.spines[["top", "right"]].set_visible(False)

# Main category distribution
ax = axes[1]
main_counts = df["main_category"].value_counts()
ax.barh(main_counts.index, main_counts.values, color="coral", edgecolor="white")
ax.set_title("Products per Main Category", fontsize=13)
ax.set_xlabel("Count")
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("ecommerce_products_5k — Dataset Overview", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## 2. Embed products

First run: calls OpenAI API (~$0.002, ~30s for 5k products).  
Subsequent runs: loads from `.cache/cb_eval/embeddings.npy` instantly.

In [ ]:
CACHE_DIR = PROJECT_ROOT / ".cache" / "cb_eval"
K_VALUES = (5, 10, 20)

t0 = time.perf_counter()
embeddings, product_ids = embed_products(
    df=df,
    api_key=api_key,
    cache_dir=CACHE_DIR,
    use_cache=True,   # set to False to force re-embedding
)
print(f"\nEmbeddings shape : {embeddings.shape}")
print(f"Time             : {time.perf_counter()-t0:.1f}s")

## 3. Sanity check — most similar products for a query

In [ ]:
id_to_row = {pid: i for i, pid in enumerate(product_ids)}
categories = df[RELEVANCE_COL].tolist()

def show_top_similar(product_id: str, top_n: int = 10):
    idx = id_to_row[product_id]
    query_row = df[df["product_id"] == product_id].iloc[0]
    print(f"Query  : [{product_id}] {query_row['name']}  |  {query_row[RELEVANCE_COL]}")
    print()

    sim = embeddings[idx] @ embeddings.T
    sim[idx] = -2.0
    top_idx = np.argsort(sim)[::-1][:top_n]

    print(f"{'#':>3}  {'product_id':>10}  {'Score':>7}  {'Same cat?':>10}  Name  (sub_category)")
    print("-" * 80)
    for rank, i in enumerate(top_idx, 1):
        pid = product_ids[i]
        row = df[df["product_id"] == pid].iloc[0]
        same = "✓" if categories[i] == query_row[RELEVANCE_COL] else "✗"
        print(f"{rank:>3}  {pid:>10}  {sim[i]:>7.4f}  {same:>10}  {row['name']}  ({categories[i]})")

# Try a Smartphone and a Skincare product
show_top_similar("PRD00001")

In [ ]:
# Try a different category
skincare_id = df[df[RELEVANCE_COL] == "Skincare"]["product_id"].iloc[0]
show_top_similar(skincare_id)

## 4. Full evaluation

In [ ]:
t0 = time.perf_counter()
results = run_evaluation(embeddings, product_ids, categories, K_VALUES)
print(f"\nTotal time: {time.perf_counter()-t0:.1f}s")
print_results(results)

In [ ]:
results_df = pd.DataFrame(
    [
        {
            "K": k,
            "Precision@K": r.precision,
            "Recall@K": r.recall,
            "HitRate@K": r.hit_rate,
            "MeanCosSim": r.mean_cosine_sim,
        }
        for k, r in sorted(results.items())
    ]
).set_index("K")

results_df.style \
    .format("{:.4f}") \
    .background_gradient(cmap="YlGn") \
    .set_caption(f"Content-Based Filtering Evaluation — {EMBED_MODEL}")

## 5. Visualisation

In [ ]:
metrics_meta = [
    ("Precision@K", "precision",       "#4C72B0"),
    ("Recall@K",    "recall",           "#DD8452"),
    ("HitRate@K",   "hit_rate",         "#55A868"),
    ("MeanCosSim",  "mean_cosine_sim",  "#C44E52"),
]

ks = sorted(results.keys())
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (label, field, color) in zip(axes, metrics_meta):
    vals = [getattr(results[k], field) for k in ks]
    bars = ax.bar([str(k) for k in ks], vals, color=color, alpha=0.85, edgecolor="white", width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=10)
    ax.set_title(label, fontsize=13, fontweight="bold")
    ax.set_xlabel("K")
    ax.set_ylim(0, min(1.0, max(vals) * 1.35 + 0.05))
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    f"Content-Based Filtering Evaluation\nModel: {EMBED_MODEL}  |  Ground truth: same sub_category",
    fontsize=13, y=1.04,
)
fig.tight_layout()
plt.show()

In [ ]:
# Line chart — all metrics vs K
fig, ax = plt.subplots(figsize=(8, 5))

for label, field, color in metrics_meta:
    vals = [getattr(results[k], field) for k in ks]
    ax.plot([str(k) for k in ks], vals, marker="o", label=label, color=color, linewidth=2, markersize=8)

ax.set_xlabel("K", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("CB Evaluation Metrics vs. K", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 6. Per-category breakdown (at K=10)

In [ ]:
K_BREAKDOWN = 10
unique_cats = sorted(df[RELEVANCE_COL].unique())
cat_to_idx  = {cat: np.where(np.array(categories) == cat)[0] for cat in unique_cats}

per_cat_results = []
for cat, indices in cat_to_idx.items():
    cat_embeddings = embeddings[indices]
    cat_ids        = [product_ids[i] for i in indices]
    cat_cats       = [categories[i] for i in indices]
    r = evaluate(
        embeddings=cat_embeddings,
        product_ids=cat_ids,
        categories=cat_cats,
        k=K_BREAKDOWN,
    )
    per_cat_results.append({"sub_category": cat, "Precision@10": r.precision, "HitRate@10": r.hit_rate, "MeanCosSim": r.mean_cosine_sim})

cat_df = pd.DataFrame(per_cat_results).sort_values("Precision@10", ascending=False).set_index("sub_category")
cat_df.style.format("{:.4f}").background_gradient(cmap="RdYlGn")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col, color in [
    (axes[0], "Precision@10", "steelblue"),
    (axes[1], "MeanCosSim",   "coral"),
]:
    sorted_df = cat_df.sort_values(col, ascending=True)
    ax.barh(sorted_df.index, sorted_df[col], color=color, edgecolor="white")
    ax.set_title(f"{col} by Sub-Category", fontsize=12, fontweight="bold")
    ax.set_xlabel(col)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Per-Category Breakdown @ K=10", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## 7. Similarity score distribution

In [ ]:
# Sample 200 products, collect similarity scores for same-cat vs diff-cat
sample_size = 200
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(product_ids), size=sample_size, replace=False)
cat_arr = np.array(categories)

same_cat_scores, diff_cat_scores = [], []
for i in sample_idx:
    sim = embeddings[i] @ embeddings.T
    sim[i] = -2.0
    mask_same = cat_arr == categories[i]
    mask_same[i] = False
    same_cat_scores.extend(sim[mask_same].tolist())
    diff_cat_scores.extend(sim[~mask_same].tolist())

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(diff_cat_scores, bins=80, alpha=0.6, label="Different sub_category", color="#4C72B0", density=True)
ax.hist(same_cat_scores, bins=80, alpha=0.6, label="Same sub_category",      color="#DD8452", density=True)
ax.set_xlabel("Cosine Similarity", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Similarity Score Distribution: Same vs Different Category", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

print(f"Same-cat mean similarity  : {np.mean(same_cat_scores):.4f}")
print(f"Diff-cat mean similarity  : {np.mean(diff_cat_scores):.4f}")
print(f"Separation (delta)        : {np.mean(same_cat_scores) - np.mean(diff_cat_scores):.4f}")

## 8. Summary & interpretation

### Reading the numbers

| Score range | Interpretation |
|---|---|
| Precision@10 > 0.6 | Strong — model reliably surfaces same-category items |
| Precision@10 0.3–0.6 | Good — clear signal above random baseline (≈5%) |
| HitRate@10 > 0.9 | Excellent — almost every product gets ≥1 relevant result |
| Mean Cosine Sim separation | Large gap between same/diff category = well-separated embedding space |

### Limitations

1. **Sub-category = relevant** is a proxy. In production, user preference matters more than taxonomy.
2. The dataset is **synthetic** (Claude-generated) — descriptions within a category may follow similar templates, inflating scores.
3. `text-embedding-3-small` was trained on broad web data — domain-specific fine-tuning could improve results further.